# Pastas del Norte · análisis de demanda

Notebook base para Google Colab. Presenta un flujo reproducible de preparación, entrenamiento y evaluación con datos de ejemplo. Sustituye el conjunto sintético por datos autorizados antes de usar resultados reales.

## 1. Contexto y alcance

El caso estudia cómo integrar ventas, producción, compras e inventario para apoyar la planificación. Los registros generados aquí son ficticios y sirven únicamente para demostrar el flujo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(7)


## 2. Crear un conjunto de ejemplo

En un entorno real, reemplaza esta celda por una lectura segura desde las réplicas A1 (histórico Oracle) y A2 (staging SQL Server), respetando permisos y minimización de datos.

In [ ]:
fechas = pd.date_range('2024-01-01', periods=180, freq='D')
tendencia = np.linspace(90, 125, len(fechas))
estacionalidad = 12 * np.sin(np.arange(len(fechas)) * 2 * np.pi / 7)
demanda = np.maximum(20, tendencia + estacionalidad + np.random.normal(0, 7, len(fechas)))
df = pd.DataFrame({'fecha': fechas, 'demanda': demanda})
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['demanda_7d'] = df['demanda'].shift(1).rolling(7).mean()
df = df.dropna().reset_index(drop=True)
df.tail()

## 3. Explorar y preparar

La separación temporal evita entrenar con observaciones futuras. Revisa calidad, faltantes y cambios de esquema antes de ejecutar este paso con datos reales.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['fecha'], df['demanda'], label='Demanda observada')
plt.title('Demanda diaria de ejemplo')
plt.xlabel('Fecha'); plt.ylabel('Unidades'); plt.legend(); plt.grid(alpha=.2)
plt.show()

features = ['dia_semana', 'mes', 'demanda_7d']
corte = int(len(df) * 0.8)
train, test = df.iloc[:corte], df.iloc[corte:]
X_train, y_train = train[features], train['demanda']
X_test, y_test = test[features], test['demanda']

## 4. Entrenar y evaluar

El modelo de bosque aleatorio es una referencia didáctica. Compara sus resultados con una línea base y valida con responsables del proceso antes de cualquier decisión operativa.

In [ ]:
modelo = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=7)
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f'MAE de ejemplo: {mae:.2f} unidades')
print(f'RMSE de ejemplo: {rmse:.2f} unidades')

plt.figure(figsize=(12, 4))
plt.plot(test['fecha'], y_test, label='Real')
plt.plot(test['fecha'], pred, label='Estimación')
plt.title('Evaluación temporal del modelo')
plt.legend(); plt.grid(alpha=.2); plt.show()

## 5. Dashboard y siguientes pasos

Usa las visualizaciones para conversar sobre demanda, producción e inventario. En la implementación real, agrega validaciones del ETL, monitoreo de deriva, control de acceso, registro de versiones y aprobación del dueño del proceso. No uses este ejemplo para compras o producción reales.

## 6. Matriz de evaluación y gobierno de riesgos

La matriz cruza los activos A1–A10 con las amenazas T1–T10. Cada celda muestra el nivel de riesgo asignado (NR = P × I); una celda vacía indica que no se registró un escenario para esa combinación. Los conteos de la leyenda se calculan desde los datos.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

# 1. Definición de Activos y Amenazas
assets = [
    "A1 Oracle",
    "A2 MS-SQL",
    "A3 Scripts ETL",
    "A4 Notebooks/Modelos",
    "A5 Datos Prod/Ventas",
    "A6 Credenciales",
    "A7 Google Colab",
    "A8 Estaciones/VS Code",
    "A9 Talento humano",
    "A10 Dashboards",
]

threats = [
    "T1 Exfiltración",
    "T2 Falla ETL",
    "T3 Inyección/Malware",
    "T4 Red",
    "T5 Error/Drift",
    "T6 Hardware",
    "T7 Fuga talento",
    "T8 Esquema",
    "T9 Cred. compartidas",
    "T10 Shadow data",
]

# 2. Matriz de Nivel de Riesgo (NR = P x I)
data = [
    [np.nan, 9, np.nan, np.nan, np.nan, 15, np.nan, 9, 20, np.nan],  # A1
    [np.nan, 16, 8, np.nan, 12, 12, np.nan, 9, 16, np.nan],  # A2
    [np.nan, 16, 12, 9, 9, np.nan, 9, 16, np.nan, np.nan],  # A3
    [12, np.nan, 12, np.nan, 20, np.nan, 12, np.nan, np.nan, 9],  # A4
    [20, 9, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 15, 20],  # A5
    [15, np.nan, 15, np.nan, np.nan, np.nan, np.nan, np.nan, 25, np.nan],  # A6
    [16, np.nan, np.nan, 12, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],  # A7
    [np.nan, np.nan, 16, 6, np.nan, 4, np.nan, np.nan, np.nan, 20],  # A8
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 20, np.nan, np.nan, np.nan],  # A9
    [6, 12, np.nan, 6, 20, np.nan, np.nan, np.nan, np.nan, np.nan],  # A10
]

df_risk = pd.DataFrame(data, index=assets, columns=threats)

# 3. Categorización para mapear colores
def get_risk_category_matrix(matrix):
    cat_matrix = np.full(matrix.shape, np.nan)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix.iloc[i, j]
            if pd.isna(val):
                continue
            elif 1 <= val <= 5:
                cat_matrix[i, j] = 1  # Bajo
            elif 6 <= val <= 11:
                cat_matrix[i, j] = 2  # Medio
            elif 12 <= val <= 19:
                cat_matrix[i, j] = 3  # Alto
            elif 20 <= val <= 25:
                cat_matrix[i, j] = 4  # Crítico
    return cat_matrix

cat_matrix = get_risk_category_matrix(df_risk)

# Paleta de colores
colors = ["#71c784", "#ffd54f", "#ff9800", "#e53935"]
cmap_custom = ListedColormap(colors)

# Conteos calculados directamente para que la leyenda coincida con la matriz
counts = {level: int(np.count_nonzero(cat_matrix == level)) for level in range(1, 5)}

# 4. Configuración del gráfico
fig, ax = plt.subplots(figsize=(10, 6.5), dpi=100)
fig.patch.set_facecolor("#ffffff")
ax.set_facecolor("#f8f9fa")
ax.imshow(cat_matrix, cmap=cmap_custom, vmin=1, vmax=4, aspect="auto")

# Rejilla blanca, incluidos los bordes exteriores
ax.set_xticks(np.arange(len(threats) + 1) - 0.5, minor=True)
ax.set_yticks(np.arange(len(assets) + 1) - 0.5, minor=True)
ax.grid(which="minor", color="#ffffff", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", size=0)

# Ejes de texto ajustados
ax.set_xticks(np.arange(len(threats)))
ax.set_yticks(np.arange(len(assets)))
ax.set_xticklabels(threats, rotation=35, ha="right", fontsize=8.5, fontweight="bold")
ax.set_yticklabels(assets, fontsize=8.5, fontweight="bold")
ax.tick_params(axis="both", length=0, pad=5)

# Números dentro de cada celda
for i in range(len(assets)):
    for j in range(len(threats)):
        val = df_risk.iloc[i, j]
        if not pd.isna(val):
            text_color = "white" if val >= 12 else "#212121"
            ax.text(j, i, f"{int(val)}", ha="center", va="center",
                    color=text_color, fontsize=9.5, fontweight="bold")

# Título y subtítulo
ax.set_title(
    "MATRIZ DE EVALUACIÓN Y GOBIERNO DE RIESGOS\n"
    "Pastas del Norte S.A. — Unidad de Inteligencia de Datos (UID)",
    fontsize=11,
    fontweight="bold",
    color="#1a237e",
    pad=12,
)

# Leyenda compacta con cantidades reales
legend_patches = [
    mpatches.Patch(color="#e53935", label=f"Crítico (20–25): {counts[4]}"),
    mpatches.Patch(color="#ff9800", label=f"Alto (12–19): {counts[3]}"),
    mpatches.Patch(color="#ffd54f", label=f"Medio (6–11): {counts[2]}"),
    mpatches.Patch(color="#71c784", label=f"Bajo (1–5): {counts[1]}"),
]
ax.legend(handles=legend_patches, bbox_to_anchor=(0.5, -0.22), loc="upper center",
          ncol=4, frameon=True, facecolor="#ffffff", edgecolor="#e0e0e0", fontsize=8.5)

plt.tight_layout()
plt.subplots_adjust(top=0.88, bottom=0.22)
plt.show()